# COM748 — Explainable ML for Heart Disease Prediction
**Dataset:** UCI Heart Disease (all 4 centres, 920 rows)  
**Models:** Logistic Regression, Random Forest, XGBoost, SVM, KNN  
**Explainability:** SHAP (global + local) + LIME (local)  
**Methodology:** CRISP-DM with stratified 5-fold CV and SMOTE inside folds

## 0. Install Dependencies

In [ ]:
!pip install -q shap lime imbalanced-learn xgboost scikit-learn pandas numpy matplotlib seaborn

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import shap
import lime
import lime.lime_tabular

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load Data
Upload `heart_disease_uci.csv` when prompted, or mount Google Drive.

In [ ]:
# Mount Google Drive — file is at MyDrive/london research/heart_disease_uci.csv
from google.colab import drive
drive.mount('/content/drive')

df_raw = pd.read_csv('/content/drive/MyDrive/london research/heart_disease_uci.csv')

print('Shape:', df_raw.shape)
df_raw.head()

## 3. Data Understanding

In [ ]:
print('=== Dataset Distribution ===')
print(df_raw['dataset'].value_counts())
print('\n=== Target Distribution (num) ===')
print(df_raw['num'].value_counts())
print('\n=== Missing Values ===')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
missing_df = pd.DataFrame({'Missing': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing'] > 0])
print('\n=== Empty strings (coded as blank) ===')
print((df_raw == '').sum())

# Warn about high-missingness columns
HIGH_MISS_THRESHOLD = 50  # percent
high_miss = missing_pct[missing_pct > HIGH_MISS_THRESHOLD]
if not high_miss.empty:
    print('\n⚠️  HIGH MISSINGNESS — imputation will be used for all columns:')
    for col, pct in high_miss.items():
        strategy = 'most_frequent' if col in ['thal', 'slope', 'fbs', 'restecg', 'exang'] else 'median'
        print(f'   {col}: {pct}% missing → {strategy} imputation')

In [ ]:
df_raw.describe(include='all')

## 4. Preprocessing

In [ ]:
df = df_raw.copy()

# Drop non-feature columns — Drive version has an extra 'index' column
drop_cols = ['id', 'dataset', 'index']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

# Replace empty strings with NaN so imputers handle them
df.replace('', np.nan, inplace=True)

# Normalise fbs/exang to uppercase TRUE/FALSE (Drive CSV uses lowercase)
for col in ['fbs', 'exang']:
    df[col] = df[col].astype(str).str.upper().replace({'NAN': np.nan, 'NONE': np.nan})

# Binarise target: 0 = no disease, 1 = disease present
df['target'] = (pd.to_numeric(df['num'], errors='coerce') > 0).astype(int)
df.drop(columns=['num'], inplace=True)

# Missing value heatmap — shows which columns/rows have gaps
plt.figure(figsize=(12, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Value Map (yellow = missing)')
plt.tight_layout()
plt.savefig('missing_values.png', dpi=150)
plt.show()

print('Target distribution:')
print(df['target'].value_counts())
print(f'\nDisease positive rate: {df["target"].mean():.1%}')
print('\nMissing after normalisation:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Outlier capping with IQR 1.5x rule
# 'ca' is excluded — it is a discrete count (0–3), IQR capping would distort it
# 'oldpeak' negative values are physically impossible — clip to 0 first
continuous_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']

df['oldpeak'] = pd.to_numeric(df['oldpeak'], errors='coerce').clip(lower=0)

fig, axes = plt.subplots(2, len(continuous_cols), figsize=(18, 6))
fig.suptitle('Outlier Capping — Before vs After (IQR 1.5x)', fontsize=13)

for i, col in enumerate(continuous_cols):
    df[col] = pd.to_numeric(df[col], errors='coerce')
    axes[0, i].boxplot(df[col].dropna())
    axes[0, i].set_title(f'{col}\nbefore')

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower, upper=upper)

    axes[1, i].boxplot(df[col].dropna())
    axes[1, i].set_title(f'{col}\nafter')

# Ensure ca is numeric (it has 66% missing but valid values are 0–3)
df['ca'] = pd.to_numeric(df['ca'], errors='coerce')

plt.tight_layout()
plt.savefig('outlier_capping.png', dpi=150)
plt.show()

print('oldpeak range after fix:', df['oldpeak'].min(), '–', df['oldpeak'].max())
print('ca range:', df['ca'].min(), '–', df['ca'].max())

In [ ]:
# Define feature columns
numeric_features   = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

X = df[numeric_features + categorical_features]
y = df['target']

# Ensure numeric types
X[numeric_features] = X[numeric_features].apply(pd.to_numeric, errors='coerce')

print('Feature matrix shape:', X.shape)
print('Missing values remaining:')
print(X.isnull().sum())

In [ ]:
# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print(f'Train disease rate: {y_train.mean():.1%} | Test disease rate: {y_test.mean():.1%}')

In [ ]:
# Preprocessing pipeline (fits only on training data)
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer,  numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

## 5. Model Training with SMOTE inside CV folds

In [ ]:
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=100, max_depth=None,
                                                   min_samples_split=2, random_state=RANDOM_STATE),
    'XGBoost':             XGBClassifier(n_estimators=100, learning_rate=0.1,
                                          max_depth=6, eval_metric='logloss',
                                          random_state=RANDOM_STATE, verbosity=0),
    'SVM':                 SVC(C=1.0, kernel='rbf', gamma='scale', probability=True,
                                random_state=RANDOM_STATE),
    'KNN':                 KNeighborsClassifier(n_neighbors=5, metric='minkowski')
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

cv_results = {}

for name, clf in classifiers.items():
    # SMOTE is inside the pipeline so it only sees training folds
    pipe = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote',        SMOTE(k_neighbors=5, random_state=RANDOM_STATE)),
        ('classifier',   clf)
    ])
    scores = cross_validate(pipe, X_train, y_train, cv=cv,
                             scoring=scoring, return_train_score=False)
    cv_results[name] = scores
    print(f'{name:22s}  AUC: {scores["test_roc_auc"].mean():.3f} ± {scores["test_roc_auc"].std():.3f}')

print('\nDone.')

In [ ]:
# Cross-validation summary table
cv_summary = []
for name, scores in cv_results.items():
    cv_summary.append({
        'Model':     name,
        'Accuracy':  f"{scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}",
        'Precision': f"{scores['test_precision'].mean():.3f} ± {scores['test_precision'].std():.3f}",
        'Recall':    f"{scores['test_recall'].mean():.3f} ± {scores['test_recall'].std():.3f}",
        'F1':        f"{scores['test_f1'].mean():.3f} ± {scores['test_f1'].std():.3f}",
        'AUC':       f"{scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f}",
    })

cv_df = pd.DataFrame(cv_summary)
cv_df

## 6. Fit Final Models on Full Training Set & Evaluate on Test Set

In [ ]:
fitted_models = {}
test_results  = []

for name, clf in classifiers.items():
    pipe = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote',        SMOTE(k_neighbors=5, random_state=RANDOM_STATE)),
        ('classifier',   clf)
    ])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe

    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    test_results.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred),
        'AUC':       roc_auc_score(y_test, y_proba),
    })

test_df = pd.DataFrame(test_results).set_index('Model')
test_df = test_df.round(3)
test_df

## 7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle('Confusion Matrices — Test Set', fontsize=13)

for ax, (name, pipe) in zip(axes, fitted_models.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Disease', 'Disease'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()

## 8. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, pipe in fitted_models.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=ax, name=name)

ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.5)')
ax.set_title('ROC Curves — All 5 Classifiers')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150)
plt.show()

## 9. SHAP — Global Interpretability (Random Forest)

In [ ]:
# Extract preprocessed feature names for SHAP labels
rf_pipe = fitted_models['Random Forest']
prep    = rf_pipe.named_steps['preprocessor']
rf_clf  = rf_pipe.named_steps['classifier']

cat_feature_names = prep.named_transformers_['cat']['encoder'].get_feature_names_out(categorical_features).tolist()
all_feature_names = numeric_features + cat_feature_names

# Transform test set (no SMOTE on test)
X_test_prep = prep.transform(X_test)

explainer   = shap.TreeExplainer(rf_clf)
shap_values = explainer.shap_values(X_test_prep)

print('Raw SHAP values type:', type(shap_values))
print('Raw SHAP values shape:', np.array(shap_values).shape)

# Handle both old API (list of 2 arrays) and new API (3D array shape: samples x features x classes)
if isinstance(shap_values, list):
    sv_positive = shap_values[1]          # old: list[class_index]
elif shap_values.ndim == 3:
    sv_positive = shap_values[:, :, 1]    # new: (samples, features, classes) → take class 1
else:
    sv_positive = shap_values             # already 2D

print('sv_positive shape (should be samples x features):', sv_positive.shape)
print('Expected: (184,', len(all_feature_names), ')')

In [ ]:
# SHAP Summary Plot (Bee Swarm) — global feature importance
# Use sv_positive which is already (samples x features) for class 1
plt.figure()
shap.summary_plot(sv_positive, X_test_prep,
                  feature_names=all_feature_names,
                  show=False, plot_type='dot')
plt.title('SHAP Summary Plot — Random Forest (Global)')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Bar Plot — mean absolute importance
plt.figure()
shap.summary_plot(sv_positive, X_test_prep,
                  feature_names=all_feature_names,
                  plot_type='bar', show=False)
plt.title('SHAP Mean Absolute Feature Importance — Random Forest')
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# Also print top 10 features by mean absolute SHAP value
mean_shap = np.abs(sv_positive).mean(axis=0)
top_features = sorted(zip(all_feature_names, mean_shap), key=lambda x: x[1], reverse=True)[:10]
print('\nTop 10 features by mean |SHAP|:')
for name, val in top_features:
    print(f'  {name:35s} {val:.4f}')

## 10. SHAP — Local Waterfall Plot (Single Patient)

In [ ]:
# Pick first true positive patient from test set
y_pred_rf = rf_pipe.predict(X_test)
pos_idx   = np.where((y_pred_rf == 1) & (y_test.values == 1))[0][0]

# Flatten expected_value to a scalar for class 1
ev = explainer.expected_value
if isinstance(ev, (list, np.ndarray)):
    base_val = float(np.array(ev).flatten()[1])
else:
    base_val = float(ev)

explanation = shap.Explanation(
    values        = sv_positive[pos_idx],
    base_values   = base_val,
    data          = X_test_prep[pos_idx],
    feature_names = all_feature_names
)

plt.figure()
shap.plots.waterfall(explanation, show=False)
plt.title(f'SHAP Waterfall — Patient {pos_idx} (True Positive)')
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. LIME — Local Explanation (Single Patient)

In [ ]:
# LIME needs a predict_proba function that works on raw (preprocessed) numpy arrays
def rf_predict_proba(X_array):
    return rf_clf.predict_proba(X_array)

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data   = prep.transform(X_train),
    feature_names   = all_feature_names,
    class_names     = ['No Disease', 'Disease'],
    mode            = 'classification',
    random_state    = RANDOM_STATE
)

lime_exp = lime_explainer.explain_instance(
    data_row        = X_test_prep[pos_idx],
    predict_fn      = rf_predict_proba,
    num_features    = 10
)

# Show as matplotlib figure and save
fig = lime_exp.as_pyplot_figure()
fig.suptitle(f'LIME Explanation — Patient {pos_idx} (True Positive)', fontsize=11)
plt.tight_layout()
plt.savefig('lime_positive.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# LIME for a true negative patient
neg_idx  = np.where((y_pred_rf == 0) & (y_test.values == 0))[0][0]

lime_exp_neg = lime_explainer.explain_instance(
    data_row   = X_test_prep[neg_idx],
    predict_fn = rf_predict_proba,
    num_features = 10
)

fig = lime_exp_neg.as_pyplot_figure()
fig.suptitle(f'LIME Explanation — Patient {neg_idx} (True Negative)', fontsize=11)
plt.tight_layout()
plt.savefig('lime_negative.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Final Summary Table

In [ ]:
print('=== Test Set Results — All 5 Models ===')
print(test_df.to_string())
print('\n=== Cross-Validation Results (mean ± std) ===')
print(cv_df.to_string(index=False))

## 13. Download All Output Files

In [ ]:
from google.colab import files

output_files = [
    'missing_values.png',
    'outlier_capping.png',
    'confusion_matrices.png',
    'roc_curves.png',
    'shap_summary.png',
    'shap_bar.png',
    'shap_waterfall.png',
    'lime_positive.png',
    'lime_negative.png',
]

for f in output_files:
    try:
        files.download(f)
    except Exception as e:
        print(f'Could not download {f}: {e}')

print('All files downloaded.')